# SATF-CNN风格的SSVEP跨域直接泛化

本 notebook 根据给定的SATF-CNN机制实现8通道、3频率版本：

```text
标准化EEG [8×372]
→ PESA：按当前trial动态计算通道Q/K，保留Top-k通道注意力
→ TCNN时域分支 + FFT频域分支
→ KAN可学习非线性映射
→ 频率分类器：15/12/10 Hz
```

实验协议：

- 只使用source session的有标签数据训练，损失只有频率交叉熵。
- 不使用DANN、GRL、域分类器或target无标签适配。
- 验证集只从source训练池内部切出，并仅用它控制学习率和早停。
- `DRIFT_FILES`在模型冻结后才加载，训练期间其EEG与标签均不可见。
- 当前`test/four`是session划分，因此本实验是严格的unseen-session评估；不能仅凭这些数据声称跨受试结果。

说明：论文完整的层宽、卷积核和KAN内部参数未包含在当前需求文本中。本实现保留PESA动态Top-k、TCNN+FFT双分支和KAN三项核心机制，并明确给出本地参数，不冒充逐参数复刻。


In [1]:
from pathlib import Path
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import resample
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


device: cpu


## 1. 严格source训练与最终测试隔离

`TRAIN_FILES`内部按“文件 × 类别”分层切出source validation。`TEST_FILES`只做source独立文件诊断；`DRIFT_FILES`是最终完全未见测试。不存在target adaptation文件。

各文件有效区间使用全部采样点并重采样到372点，以满足批处理长度一致；不补零，也不依据最终测试结果选择长度。


In [2]:
DATA_DIR = Path("data")

TRAIN_FILES = [f"test_{index}_ssvep" for index in range(2, 8)]
TEST_FILES = ["test_8_ssvep"]
DRIFT_FILES = ["four_5_ssvep"]

FREQUENCIES_HZ = np.array([15.0, 12.0, 10.0])
SAMPLE_RATE_HZ = 250.0
FOUR_DELAY_SECONDS = 0.5
FOUR_WINDOW_SECONDS = 1.5
MODEL_SAMPLES = 372
EEG_CHANNELS = 8
VALIDATION_FRACTION = 0.20

ATTENTION_DIM = 32
TOPK_RATIOS = (0.50, 2 / 3, 0.75)
TCNN_FEATURES = 64
FFT_FEATURES = 32
TCNN_POOL_BINS = 4
FFT_POOL_BINS = 8
FUSION_FEATURES = TCNN_FEATURES * TCNN_POOL_BINS + FFT_FEATURES * FFT_POOL_BINS
KAN_HIDDEN = 64
KAN_EMBEDDING = 32
KAN_GRID_SIZE = 8
KAN_SPLINE_ORDER = 3
DROPOUT = 0.40

BATCH_SIZE = 64
MAX_EPOCHS = 500
EARLY_STOP_PATIENCE = 60
LR_PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


def title(file_names):
    return "/".join(name.removesuffix("_ssvep") for name in file_names)


all_roles = TRAIN_FILES + TEST_FILES + DRIFT_FILES
assert len(all_roles) == len(set(all_roles)), "file leakage across roles"
assert all(name.startswith("test_") for name in TRAIN_FILES + TEST_FILES)
assert all(name.startswith("four_") for name in DRIFT_FILES)
print("file roles")
print("  source train/internal validation:", title(TRAIN_FILES))
print("  held-out source diagnostic:", title(TEST_FILES))
print("  final unseen-session test:", title(DRIFT_FILES))
print("  target adaptation: NONE")


file roles
  source train/internal validation: test_2/test_3/test_4/test_5/test_6/test_7
  held-out source diagnostic: test_8
  final unseen-session test: four_5
  target adaptation: NONE


In [3]:
def prepare_trials(raw, name):
    x = np.stack(raw) if np.asarray(raw).dtype == object else np.asarray(raw)
    if x.ndim != 3 or x.shape[1] != EEG_CHANNELS:
        raise ValueError(f"{name}: expected [trial,{EEG_CHANNELS},time], got {x.shape}")
    if name.startswith("four_"):
        start = round(FOUR_DELAY_SECONDS * SAMPLE_RATE_HZ)
        requested = round(FOUR_WINDOW_SECONDS * SAMPLE_RATE_HZ)
        if x.shape[-1] <= start:
            raise ValueError(f"{name}: cannot skip {start} samples from {x.shape[-1]}")
        x = x[:, :, start:min(start + requested, x.shape[-1])]
    elif not name.startswith("test_"):
        raise ValueError(f"{name}: unknown session prefix")

    original_samples = x.shape[-1]
    if original_samples != MODEL_SAMPLES:
        x = resample(x, MODEL_SAMPLES, axis=-1)
    x = np.asarray(x, dtype=np.float32)
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    x = (x - mean) / np.maximum(std, 1e-6)
    return x, original_samples


def load_labeled_files(file_names):
    records = []
    for name in file_names:
        x, original_samples = prepare_trials(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        y = np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)
        if len(x) != len(y):
            raise ValueError(f"{name}: trial/label length mismatch")
        if not np.isin(y, np.arange(len(FREQUENCIES_HZ))).all():
            raise ValueError(f"{name}: labels must be 0..{len(FREQUENCIES_HZ)-1}")
        records.append((x, y, name))
        print(
            "labeled", name, "original_samples", original_samples,
            "model_shape", x.shape,
            "counts", np.bincount(y, minlength=len(FREQUENCIES_HZ)),
        )
    return records


def concatenate_labeled(records):
    return (
        np.concatenate([x for x, _, _ in records]),
        np.concatenate([y for _, y, _ in records]),
        np.concatenate([np.full(len(y), name, dtype=object) for _, y, name in records]),
    )


# Before training, load source training files only.
source_records = load_labeled_files(TRAIN_FILES)
X_pool, y_pool, source_pool = concatenate_labeled(source_records)
indices = np.arange(len(y_pool))
strata = np.array([f"{source}::{label}" for source, label in zip(source_pool, y_pool)])
train_indices, validation_indices = train_test_split(
    indices, test_size=VALIDATION_FRACTION, random_state=SEED, stratify=strata
)
assert set(train_indices).isdisjoint(validation_indices)
X_train, y_train = X_pool[train_indices], y_pool[train_indices]
X_validation, y_validation = X_pool[validation_indices], y_pool[validation_indices]
print("source train / internal validation:", len(y_train), len(y_validation))
print("final test files have not been loaded")


labeled test_2_ssvep original_samples 374 model_shape (19, 8, 372) counts [8 4 7]
labeled test_3_ssvep original_samples 373 model_shape (41, 8, 372) counts [11 16 14]
labeled test_4_ssvep original_samples 374 model_shape (69, 8, 372) counts [17 23 29]
labeled test_5_ssvep original_samples 373 model_shape (102, 8, 372) counts [30 35 37]
labeled test_6_ssvep original_samples 373 model_shape (106, 8, 372) counts [17 43 46]
labeled test_7_ssvep original_samples 373 model_shape (67, 8, 372) counts [19 20 28]
source train / internal validation: 323 81
final test files have not been loaded


## 2. PESA动态Top-k通道注意力

每个trial独立计算通道Q/K注意力。对每个query通道分别保留Top-k key通道，其余权重置零；本实现使用$C/2$、$2C/3$、$3C/4$三种稀疏度并学习融合权重。

因此选择结果由当前trial决定，不是预先固定某几个电极。输出保留原始时间轴，供TCNN与FFT分支继续处理。


In [4]:
class DynamicSinusoidalPositionEncoding(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        position = torch.arange(samples, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, channels, 2, dtype=torch.float32)
            * (-math.log(10000.0) / channels)
        )
        encoding = torch.zeros(samples, channels, dtype=torch.float32)
        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term[:encoding[:, 1::2].shape[1]])
        self.register_buffer("encoding", encoding.T.unsqueeze(0))
        self.gate = nn.Sequential(
            nn.Linear(channels * 2, channels), nn.Sigmoid()
        )

    def forward(self, eeg):
        statistics = torch.cat([eeg.mean(dim=-1), eeg.std(dim=-1)], dim=1)
        gate = self.gate(statistics).unsqueeze(-1)
        return eeg + gate * self.encoding


class PESATopKChannelAttention(nn.Module):
    def __init__(self, channels, samples, attention_dim, ratios):
        super().__init__()
        self.channels = channels
        self.samples = samples
        self.k_values = tuple(sorted(set(
            max(1, min(channels, int(math.floor(channels * ratio))))
            for ratio in ratios
        )))
        self.query = nn.Linear(samples, attention_dim, bias=False)
        self.key = nn.Linear(samples, attention_dim, bias=False)
        self.value = nn.Linear(samples, samples, bias=False)
        self.fusion_logits = nn.Parameter(torch.zeros(len(self.k_values)))
        self.output_norm = nn.LayerNorm(samples)

    def sparse_attention(self, scores, values, k):
        indices = scores.topk(k, dim=-1).indices
        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask.scatter_(-1, indices, True)
        sparse_scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(sparse_scores, dim=-1)
        return weights @ values, indices

    def forward(self, eeg, return_indices=False):
        q = self.query(eeg)
        k = self.key(eeg)
        v = self.value(eeg)
        scores = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1])
        outputs, selections = [], []
        for topk in self.k_values:
            output, indices = self.sparse_attention(scores, v, topk)
            outputs.append(output)
            selections.append(indices)
        mixture = torch.softmax(self.fusion_logits, dim=0)
        fused = sum(weight * output for weight, output in zip(mixture, outputs))
        enhanced = self.output_norm(eeg + fused)
        return (enhanced, selections) if return_indices else enhanced


## 3. TCNN + FFT双分支与KAN

- TCNN分支从PESA增强后的时域EEG提取局部和长程时间特征。
- FFT分支对同一PESA输出计算幅度谱，再用卷积提取频域特征。
- 两个分支融合后进入KAN。KAN层对每个输入维度学习一元B样条函数，再求和得到输出，最后由线性频率分类器给出3类结果。


In [5]:
class KANLinear(nn.Module):
    def __init__(
        self, in_features, out_features, grid_size=8, spline_order=3,
        grid_min=-3.0, grid_max=3.0,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order
        step = (grid_max - grid_min) / grid_size
        knots = (
            torch.arange(-spline_order, grid_size + spline_order + 1, dtype=torch.float32)
            * step + grid_min
        )
        self.register_buffer("knots", knots)
        self.grid_min = grid_min
        self.grid_max = grid_max
        self.base = nn.Linear(in_features, out_features)
        basis_count = grid_size + spline_order
        self.spline_weight = nn.Parameter(
            torch.empty(out_features, in_features, basis_count)
        )
        nn.init.normal_(self.spline_weight, mean=0.0, std=0.02)

    def b_spline_basis(self, x):
        x = self.grid_max * torch.tanh(x / self.grid_max)
        x = x.unsqueeze(-1)
        knots = self.knots
        basis = ((x >= knots[:-1]) & (x < knots[1:])).to(x.dtype)
        for degree in range(1, self.spline_order + 1):
            left_denominator = knots[degree:-1] - knots[:-(degree + 1)]
            right_denominator = knots[degree + 1:] - knots[1:-degree]
            left = (
                (x - knots[:-(degree + 1)])
                / left_denominator.clamp_min(torch.finfo(x.dtype).eps)
            ) * basis[..., :-1]
            right = (
                (knots[degree + 1:] - x)
                / right_denominator.clamp_min(torch.finfo(x.dtype).eps)
            ) * basis[..., 1:]
            basis = left + right
        return basis

    def forward(self, x):
        base_output = self.base(F.silu(x))
        basis = self.b_spline_basis(x)
        spline_output = torch.einsum("bin,oin->bo", basis, self.spline_weight)
        return base_output + spline_output


class SATFStyleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.position_encoding = DynamicSinusoidalPositionEncoding(
            EEG_CHANNELS, MODEL_SAMPLES
        )
        self.pesa = PESATopKChannelAttention(
            EEG_CHANNELS, MODEL_SAMPLES, ATTENTION_DIM, TOPK_RATIOS
        )
        self.tcnn = nn.Sequential(
            nn.Conv1d(EEG_CHANNELS, 32, kernel_size=9, padding=4, bias=False),
            nn.BatchNorm1d(32), nn.ELU(),
            nn.MaxPool1d(4), nn.Dropout(DROPOUT),
            nn.Conv1d(32, TCNN_FEATURES, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(TCNN_FEATURES), nn.ELU(),
            nn.AdaptiveAvgPool1d(TCNN_POOL_BINS),
        )
        fft_bins = MODEL_SAMPLES // 2 + 1
        self.fft_branch = nn.Sequential(
            nn.Conv1d(EEG_CHANNELS, FFT_FEATURES, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(FFT_FEATURES), nn.ELU(),
            nn.MaxPool1d(4), nn.Dropout(DROPOUT),
            nn.Conv1d(FFT_FEATURES, FFT_FEATURES, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(FFT_FEATURES), nn.ELU(),
            nn.AdaptiveAvgPool1d(FFT_POOL_BINS),
        )
        self.fusion_norm = nn.LayerNorm(FUSION_FEATURES)
        self.kan1 = KANLinear(
            FUSION_FEATURES, KAN_HIDDEN, KAN_GRID_SIZE, KAN_SPLINE_ORDER
        )
        self.kan2 = KANLinear(
            KAN_HIDDEN, KAN_EMBEDDING, KAN_GRID_SIZE, KAN_SPLINE_ORDER
        )
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(KAN_EMBEDDING, len(FREQUENCIES_HZ))

    def forward(self, eeg):
        positioned = self.position_encoding(eeg)
        enhanced = self.pesa(positioned)
        time_features = self.tcnn(enhanced).flatten(1)
        spectrum = torch.log1p(torch.abs(torch.fft.rfft(enhanced, dim=-1)))
        frequency_features = self.fft_branch(spectrum).flatten(1)
        fused = self.fusion_norm(torch.cat([time_features, frequency_features], dim=1))
        hidden = F.silu(self.kan1(fused))
        embedding = F.silu(self.kan2(self.dropout(hidden)))
        return self.classifier(self.dropout(embedding))

    @torch.no_grad()
    def selected_channels(self, eeg):
        self.eval()
        positioned = self.position_encoding(eeg)
        _, selections = self.pesa(positioned, return_indices=True)
        return [selection.cpu().numpy() for selection in selections]


model = SATFStyleCNN().to(DEVICE)
print(model)
print("PESA top-k values:", model.pesa.k_values)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))


SATFStyleCNN(
  (position_encoding): DynamicSinusoidalPositionEncoding(
    (gate): Sequential(
      (0): Linear(in_features=16, out_features=8, bias=True)
      (1): Sigmoid()
    )
  )
  (pesa): PESATopKChannelAttention(
    (query): Linear(in_features=372, out_features=32, bias=False)
    (key): Linear(in_features=372, out_features=32, bias=False)
    (value): Linear(in_features=372, out_features=372, bias=False)
    (output_norm): LayerNorm((372,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (tcnn): Sequential(
    (0): Conv1d(8, 32, kernel_size=(9,), stride=(1,), padding=(4,), bias=False)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.4, inplace=False)
    (5): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True

## 4. 仅用source训练

优化器为Adam，损失为三分类交叉熵。学习率调度和早停只读取source内部验证准确率；最终`DRIFT_FILES`不参与任何训练决策。


In [6]:
train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
)
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE
)


@torch.no_grad()
def predict(eeg):
    model.eval()
    tensor = torch.tensor(eeg, dtype=torch.float32, device=DEVICE)
    return model(tensor).argmax(1).cpu().numpy()


best_validation_accuracy = -1.0
best_state = None
epochs_without_improvement = 0
history = {"train": [], "source_validation": [], "learning_rate": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    correct = total = 0
    for batch_eeg, batch_labels in train_loader:
        batch_eeg = batch_eeg.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        logits = model(batch_eeg)
        loss = F.cross_entropy(logits, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        correct += (logits.argmax(1) == batch_labels).sum().item()
        total += len(batch_labels)

    train_accuracy = correct / total
    validation_prediction = predict(X_validation)
    validation_accuracy = float(np.mean(validation_prediction == y_validation))
    scheduler.step(validation_accuracy)
    current_lr = optimizer.param_groups[0]["lr"]
    history["train"].append(train_accuracy)
    history["source_validation"].append(validation_accuracy)
    history["learning_rate"].append(current_lr)

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epoch == 1 or epoch % 20 == 0:
        print(
            f"epoch={epoch:03d} train_acc={train_accuracy:.4f} "
            f"source_val_acc={validation_accuracy:.4f} lr={current_lr:.2e}"
        )
    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print("early stop at epoch", epoch)
        break

assert best_state is not None
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()
print("best source internal validation accuracy:", best_validation_accuracy)
print("final unseen-session files have not been loaded")


epoch=001 train_acc=0.3715 source_val_acc=0.3951 lr=1.00e-03
epoch=020 train_acc=0.9876 source_val_acc=0.3951 lr=1.00e-03
epoch=040 train_acc=0.9969 source_val_acc=0.4321 lr=1.00e-03
epoch=060 train_acc=0.9721 source_val_acc=0.3580 lr=5.00e-04
epoch=080 train_acc=1.0000 source_val_acc=0.4568 lr=2.50e-04
early stop at epoch 97
best source internal validation accuracy: 0.48148148148148145
final unseen-session files have not been loaded


## 5. 模型冻结后加载独立测试

模型选择和学习率调度已经结束。现在才加载source独立诊断文件和最终unseen-session文件，并一次性报告结果。


In [7]:
for parameter in model.parameters():
    parameter.requires_grad_(False)
model.eval()
print("model frozen; loading held-out files now")

source_test_records = load_labeled_files(TEST_FILES)
drift_test_records = load_labeled_files(DRIFT_FILES)
X_source_test, y_source_test, _ = concatenate_labeled(source_test_records)
X_drift_test, y_drift_test, _ = concatenate_labeled(drift_test_records)


def report(name, labels, prediction):
    correct = int(np.sum(prediction == labels))
    accuracy = correct / len(labels)
    balanced = balanced_accuracy_score(labels, prediction)
    recalls = recall_score(
        labels, prediction, labels=np.arange(len(FREQUENCIES_HZ)),
        average=None, zero_division=0,
    )
    matrix = confusion_matrix(labels, prediction, labels=np.arange(len(FREQUENCIES_HZ)))
    print(f"\n{name}")
    print(f"accuracy: {accuracy:.4f} ({correct}/{len(labels)})")
    print(f"balanced accuracy: {balanced:.4f}")
    for frequency, value in zip(FREQUENCIES_HZ, recalls):
        print(f"recall {frequency:g} Hz: {value:.4f}")
    print("confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]")
    print(matrix)
    return accuracy


source_prediction = predict(X_source_test)
drift_prediction = predict(X_drift_test)
report(f"{title(TEST_FILES)} SATF-style source diagnostic", y_source_test, source_prediction)
report(f"{title(DRIFT_FILES)} SATF-style unseen-session test", y_drift_test, drift_prediction)

sample = torch.tensor(X_drift_test[:3], dtype=torch.float32, device=DEVICE)
selections = model.selected_channels(sample)
print("\nDynamic PESA selected channel indices for first 3 final-test trials")
for topk, indices in zip(model.pesa.k_values, selections):
    print(f"top-{topk} shape={indices.shape}")
    for trial_index in range(len(sample)):
        print(f"  trial {trial_index + 1}: per-query selections={indices[trial_index].tolist()}")


model frozen; loading held-out files now
labeled test_8_ssvep original_samples 373 model_shape (67, 8, 372) counts [23 24 20]
labeled four_5_ssvep original_samples 372 model_shape (125, 8, 372) counts [25 44 56]

test_8 SATF-style source diagnostic
accuracy: 0.4328 (29/67)
balanced accuracy: 0.4350
recall 15 Hz: 0.5217
recall 12 Hz: 0.3333
recall 10 Hz: 0.4500
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[12  8  3]
 [ 7  8  9]
 [ 8  3  9]]

four_5 SATF-style unseen-session test
accuracy: 0.2800 (35/125)
balanced accuracy: 0.3509
recall 15 Hz: 0.7200
recall 12 Hz: 0.1364
recall 10 Hz: 0.1964
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[18  1  6]
 [35  6  3]
 [39  6 11]]

Dynamic PESA selected channel indices for first 3 final-test trials
top-4 shape=(3, 8, 4)
  trial 1: per-query selections=[[0, 3, 1, 7], [0, 3, 1, 5], [0, 3, 5, 4], [0, 3, 1, 5], [5, 3, 4, 0], [0, 3, 1, 5], [3, 0, 1, 5], [0, 1, 3, 5]]
  trial 2: per-query selections=[[7

## 6. 报告边界

- 训练只使用source数据和频率交叉熵，没有DANN或target adaptation。
- source validation用于学习率调度和早停；最终测试不参与任何训练决策。
- PESA选择由每个trial的输入动态产生，不是固定电极列表。
- 当前`test/four`是session而非已确认的不同受试者，因此只能报告unseen-session结果。
- 若要做严格跨受试实验，必须把不同受试者ID按组切分，并在source受试者内部另切validation。
- 任何最终测试结果都不得用于回头修改窗口、网络、随机种子或超参数。
